# Notebook 3: Inter-Event Time Regime Classification

This notebook characterizes the temporal structure of **total seismicity** (including aftershocks) in the USGS ComCat catalog (2000--2025, M >= 2.5) by fitting parametric inter-event time (IET) distributions to 2°×2° grid cells globally.

**Important caveat:** The catalog has **not** been declustered. Aftershock sequences produce bursts of short inter-event times that dominate IET distributions and drive most cells toward clustered behavior (Weibull k << 1). The results therefore characterize the composite temporal structure of mainshock-aftershock sequences, not the background seismicity regime that would emerge after removing dependent events.

**Methods overview:**
- Four candidate distributions are fit to each cell's IET series via MLE with a fixed location parameter (`floc=0`): exponential (1 free parameter: scale), gamma (2: shape, scale), log-normal (2: shape, scale), and Weibull (2: shape, scale).
- Model selection uses AIC. The classification scheme based on the Weibull shape parameter k is: **clustered** (k < 0.9), **Poisson-like** (0.9 <= k <= 1.1), **quasi-periodic** (k > 1.1), or **ambiguous** (no model achieves delta-AIC > 10 over its nearest competitor).
- Kolmogorov-Smirnov goodness-of-fit tests assess whether the best-fit distribution adequately describes each cell's IET data.
- Spearman rank correlations and nonparametric group tests (Kruskal-Wallis with post-hoc Mann-Whitney U) quantify relationships between the Weibull k parameter and geophysical covariates (b-value, depth, strain rate, tectonic setting).

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.interevent import compute_interevent_times, fit_distributions, classify_regime
from src.spatial import assign_cells
from src.gutenberg_richter import compute_bvalue_grid
from src.plotting import setup_style, save_figure, plot_regime_map
from src.external_data import (
    load_gsrm_strain, resample_strain_to_grid,
    load_pb2002_boundaries, classify_grid_tectonic_settings
)

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} events")


Loaded 681,450 events


## 3.1 Spatial Gridding and IET Computation

Events are assigned to 2°×2° grid cells. For each cell with at least 100 events and a median IET under 30 days, we compute inter-event times in hours and fit four distributions (exponential, gamma, log-normal, Weibull) using MLE with `floc=0`. The Weibull shape parameter k determines the regime classification, and the best-fit model is selected by AIC. A KS goodness-of-fit test is run on each cell's best-fit distribution to assess model adequacy.

In [3]:
df_cells = assign_cells(df, cell_size=2.0)

results = []
cell_fit_data = {}  # Store per-cell IET and fit params for goodness-of-fit tests
example_cells = {"poisson": None, "clustered": None, "quasi_periodic": None, "ambiguous": None}

for (clat, clon), grp in df_cells.groupby(["cell_lat", "cell_lon"]):
    if len(grp) < 100:
        continue

    times_sorted = grp.sort_values("time")["time"].values
    iet = compute_interevent_times(times_sorted)
    iet_hours = iet / 3600.0  # Convert to hours
    iet_hours = iet_hours[iet_hours > 0]

    if len(iet_hours) < 50:
        continue

    median_iet_days = np.median(iet_hours) / 24.0
    if median_iet_days > 30:
        continue

    try:
        fit_result = fit_distributions(iet_hours)
        regime = classify_regime(fit_result)
    except Exception:
        continue

    # Get Weibull k if available
    weibull_k = fit_result["weibull"]["params"][0]

    results.append({
        "cell_lat": clat,
        "cell_lon": clon,
        "regime": regime,
        "best_aic": fit_result["best_aic"],
        "weibull_k": weibull_k,
        "n_events": len(grp),
        "median_iet_hours": np.median(iet_hours),
    })

    # Store per-cell data for goodness-of-fit testing
    cell_fit_data[(clat, clon)] = {
        "iet": iet_hours,
        "best_aic": fit_result["best_aic"],
        "fit_result": fit_result,
    }

    # Save example cells
    if example_cells[regime] is None:
        example_cells[regime] = {"iet_hours": iet_hours, "fit_result": fit_result,
                                  "lat": clat, "lon": clon, "n": len(grp)}

regime_df = pd.DataFrame(results)
print(f"Classified {len(regime_df)} cells")
print(regime_df["regime"].value_counts())
print(f"\nOverall regime fractions:")
for r in ["clustered", "poisson", "quasi_periodic", "ambiguous"]:
    count = (regime_df["regime"] == r).sum()
    print(f"  {r:15s}: {count:4d} ({count/len(regime_df)*100:.1f}%)")

# Goodness-of-fit: KS test for each cell's best-fit distribution
n_good_fit = 0
n_poor_fit = 0
for cell_key, cdata in cell_fit_data.items():
    best = cdata["best_aic"]
    fit = cdata["fit_result"]
    dist_obj = {"exponential": stats.expon, "gamma": stats.gamma,
                "lognormal": stats.lognorm, "weibull": stats.weibull_min}[best]
    ks_stat, ks_p = stats.kstest(cdata["iet"], dist_obj.cdf, args=fit[best]["params"])
    if ks_p >= 0.05:
        n_good_fit += 1
    else:
        n_poor_fit += 1
print(f"\nGoodness-of-fit (KS test, alpha=0.05): {n_good_fit} adequate fits, "
      f"{n_poor_fit} poor fits out of {n_good_fit + n_poor_fit} cells")

Classified 715 cells
regime
clustered    528
ambiguous    187
Name: count, dtype: int64

Overall regime fractions:
  clustered      :  528 (73.8%)
  poisson        :    0 (0.0%)
  quasi_periodic :    0 (0.0%)
  ambiguous      :  187 (26.2%)



Goodness-of-fit (KS test, alpha=0.05): 347 adequate fits, 368 poor fits out of 715 cells


## 3.2 Global Regime Map

Map of regime classifications for each grid cell. Because the catalog is not declustered, the map is expected to be dominated by clustered cells (red), with very few Poisson-like or quasi-periodic cells.

In [4]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_regime_map(regime_df["cell_lat"], regime_df["cell_lon"], regime_df["regime"], ax=ax)
save_figure(fig, "03_regime_classification_map")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/3266278282.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.3 Weibull k Map

Spatial distribution of the Weibull shape parameter k. Values below 1 indicate clustered (aftershock-dominated) temporal behavior; values near 1 indicate Poisson-like arrivals; values above 1 would indicate quasi-periodic behavior. In a non-declustered catalog, nearly all cells are expected to show k well below 1.

In [5]:
from src.plotting import plot_global_map

fig, ax = plt.subplots(figsize=(14, 7))
plot_global_map(
    regime_df["cell_lat"], regime_df["cell_lon"], regime_df["weibull_k"],
    cmap="coolwarm", vmin=0.5, vmax=1.5,
    label="Weibull shape parameter k",
    title="Weibull k Parameter (k<1: clustered, k=1: Poisson, k>1: quasi-periodic)",
    ax=ax
)
save_figure(fig, "03_weibull_k_map")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/936925768.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.4 Example Distributions

Empirical and fitted CDFs for one example cell from each regime category. The quasi-periodic panel may be empty (no examples found) given the dominance of aftershock clustering in the non-declustered catalog. AIC values for each candidate distribution are shown in the legend.

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes_flat = axes.flatten()

for i, (regime_name, label) in enumerate([
    ("poisson", "Poisson-like"), ("clustered", "Clustered-like"),
    ("quasi_periodic", "Quasi-periodic-like"), ("ambiguous", "Ambiguous")
]):
    ax = axes_flat[i]
    ex = example_cells.get(regime_name)
    if ex is None:
        ax.set_title(f"{label} — no example found")
        continue

    iet = ex["iet_hours"]
    fit = ex["fit_result"]

    # Empirical CDF
    sorted_iet = np.sort(iet)
    ecdf = np.arange(1, len(sorted_iet) + 1) / len(sorted_iet)
    ax.plot(sorted_iet, ecdf, "k-", linewidth=1.5, label="Empirical CDF")

    # Fitted CDFs
    x = np.linspace(sorted_iet[0], np.percentile(sorted_iet, 99), 200)
    colors = {"exponential": "#457B9D", "gamma": "#E63946",
              "lognormal": "#F4A261", "weibull": "#2A9D8F"}
    for dist_name, color in colors.items():
        params = fit[dist_name]["params"]
        dist = getattr(stats, "expon" if dist_name == "exponential" else
                       "lognorm" if dist_name == "lognormal" else
                       "weibull_min" if dist_name == "weibull" else "gamma")
        cdf_vals = dist.cdf(x, *params)
        aic = fit[dist_name]["aic"]
        ax.plot(x, cdf_vals, color=color, linewidth=1, alpha=0.8,
                label=f"{dist_name} (AIC={aic:.0f})")

    ax.set_xlabel("Inter-event time (hours)")
    ax.set_ylabel("CDF")
    ax.set_title(f"{label} (lat={ex['lat']:.0f}, lon={ex['lon']:.0f})")
    ax.legend(fontsize=7)
    ax.set_xscale("log")

fig.suptitle("Example IET Distributions by Regime", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "03_example_distributions")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/2259060495.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.5 Weibull k vs. b-value

Scatter plot of Weibull k against Gutenberg-Richter b-value for each grid cell, colored by regime. A Spearman rank correlation tests for a monotonic relationship between temporal clustering and frequency-magnitude scaling.

In [7]:
bvalue_grid = compute_bvalue_grid(df, cell_size=2.0, min_events=50)
merged = regime_df.merge(bvalue_grid[["cell_lat", "cell_lon", "b_value"]],
                         on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
color_map = {"poisson": "#457B9D", "clustered": "#E63946",
             "quasi_periodic": "#2A9D8F", "ambiguous": "#AAAAAA"}
for regime_name in color_map:
    subset = merged[merged["regime"] == regime_name]
    ax.scatter(subset["b_value"], subset["weibull_k"], c=color_map[regime_name],
               s=20, alpha=0.5, label=regime_name.replace("_", " ").title())

ax.set_xlabel("b-value")
ax.set_ylabel("Weibull k")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Weibull Shape Parameter vs. b-value")
ax.legend()
save_figure(fig, "03_k_vs_bvalue")
plt.show()

# Spearman correlation: k vs b-value
b_values = merged["b_value"].values
k_values = merged["weibull_k"].values
rho, p_val = stats.spearmanr(b_values, k_values)
print(f"Spearman rho (k vs b-value) = {rho:.3f}, p = {p_val:.2e}, n = {len(k_values)}")

Spearman rho (k vs b-value) = 0.084, p = 3.36e-02, n = 640


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/1330122078.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.6 Weibull k vs. Depth

Scatter plot of Weibull k against median event depth for each grid cell, with point color scaled by event count. A Spearman rank correlation tests whether shallower or deeper seismogenic zones exhibit different degrees of temporal clustering.

In [8]:
df_cells_depth = assign_cells(df, cell_size=2.0)
depth_med = df_cells_depth.groupby(["cell_lat", "cell_lon"])["depth"].median().reset_index()
depth_med.columns = ["cell_lat", "cell_lon", "median_depth"]

k_depth = regime_df.merge(depth_med, on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(k_depth["median_depth"], k_depth["weibull_k"],
                c=k_depth["n_events"], cmap="viridis", s=15, alpha=0.5,
                norm=plt.matplotlib.colors.LogNorm())
plt.colorbar(sc, ax=ax, label="Number of events")
ax.set_xlabel("Median Depth (km)")
ax.set_ylabel("Weibull k")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Weibull k vs. Predominant Event Depth")
save_figure(fig, "03_k_vs_depth")
plt.show()

# Spearman correlation: k vs depth
depth_values = k_depth["median_depth"].values
k_values = k_depth["weibull_k"].values
rho, p_val = stats.spearmanr(depth_values, k_values)
print(f"Spearman rho (k vs depth) = {rho:.3f}, p = {p_val:.2e}, n = {len(depth_values)}")

Spearman rho (k vs depth) = 0.619, p = 8.76e-77, n = 715


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/1854622942.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.7 Regime vs. Geodetic Strain Rate

We join the regime classification with the GSRM v2.1 geodetic strain rate model to test whether higher tectonic deformation rates correspond to different temporal clustering patterns. Because almost all cells are classified as clustered in the non-declustered catalog, we focus on the continuous Weibull k parameter rather than regime categories. A Spearman correlation between log10(strain rate) and k quantifies the relationship.

In [9]:
# Load and resample GSRM strain rates
strain_raw = load_gsrm_strain()
strain_grid = resample_strain_to_grid(strain_raw, grid_size=2.0)

# Join with regime classification
regime_strain = regime_df.merge(
    strain_grid, left_on=["cell_lat", "cell_lon"], right_on=["lat_bin", "lon_bin"], how="inner"
)
regime_strain = regime_strain[regime_strain["mean_second_invariant"] > 0]
print(f"Matched cells (regime + strain): {len(regime_strain)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Strain rate distribution by regime
ax = axes[0]
color_map = {"poisson": "#457B9D", "clustered": "#E63946",
             "quasi_periodic": "#2A9D8F", "ambiguous": "#AAAAAA"}
for regime_name in ["poisson", "clustered", "ambiguous"]:
    subset = regime_strain[regime_strain["regime"] == regime_name]
    if len(subset) > 0:
        ax.hist(np.log10(subset["mean_second_invariant"]), bins=20, alpha=0.5,
                color=color_map[regime_name], label=f"{regime_name} (n={len(subset)})")
ax.set_xlabel("log₁₀ Strain Rate (nanostr/yr)")
ax.set_ylabel("Count")
ax.set_title("Strain Rate Distribution by Regime")
ax.legend()

# Panel 2: Weibull k vs strain rate
ax = axes[1]
from scipy.stats import spearmanr
sc = ax.scatter(regime_strain["mean_second_invariant"], regime_strain["weibull_k"],
                c=[color_map[r] for r in regime_strain["regime"]], s=15, alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel("Geodetic Strain Rate (nanostr/yr)")
ax.set_ylabel("Weibull k")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Temporal Clustering vs. Strain Rate")

rho, p = spearmanr(np.log10(regime_strain["mean_second_invariant"]), regime_strain["weibull_k"])
ax.text(0.05, 0.95, f"Spearman ρ = {rho:.3f}\np = {p:.2e}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.tight_layout()
save_figure(fig, "03_regime_vs_strain_rate")
plt.show()

# Median strain rate by regime
print("\nMedian strain rate by regime:")
for regime_name in ["poisson", "clustered", "ambiguous"]:
    subset = regime_strain[regime_strain["regime"] == regime_name]
    if len(subset) > 0:
        print(f"  {regime_name:15s}: {subset['mean_second_invariant'].median():.1f} nanostr/yr (n={len(subset)})")

Matched cells (regime + strain): 669


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/1585445899.py:44: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.tight_layout()
/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/1585445899.py:44: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.tight_layout()


/Users/christopherortiz/Desktop/memory-of-the-earth/src/plotting.py:87: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.savefig(path, dpi=DEFAULT_DPI, bbox_inches="tight")
/Users/christopherortiz/Desktop/memory-of-the-earth/src/plotting.py:87: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=DEFAULT_DPI, bbox_inches="tight")



Median strain rate by regime:
  clustered      : 131.5 nanostr/yr (n=495)
  ambiguous      : 76.0 nanostr/yr (n=174)


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/1585445899.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.8 Regime vs. Tectonic Setting (PB2002)

Using the PB2002 plate boundary model, we classify each grid cell by tectonic setting (subduction, convergent, transform, spreading, rift, intraplate) and test whether the Weibull k distribution varies systematically across settings. A Kruskal-Wallis test assesses the omnibus hypothesis of equal k distributions, followed by post-hoc pairwise Mann-Whitney U tests with Bonferroni correction. Given the aftershock-dominated catalog, all settings are expected to show predominantly clustered behavior, but subtle differences in the degree of clustering (median k) may still emerge.

In [10]:
# Classify regime grid cells by tectonic setting
boundaries = load_pb2002_boundaries()
tec_settings = classify_grid_tectonic_settings(
    regime_df["cell_lat"].values, regime_df["cell_lon"].values,
    boundaries, radius_deg=2.0
)
regime_df["tectonic_setting"] = tec_settings

setting_order = ["subduction", "convergent", "transform", "spreading", "rift", "intraplate"]
setting_colors = {
    "subduction": "#E63946", "convergent": "#F4A261", "transform": "#457B9D",
    "spreading": "#2A9D8F", "rift": "#A8DADC", "intraplate": "#AAAAAA"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: Regime distribution by tectonic setting (stacked bar)
ax = axes[0]
regime_names = ["clustered", "ambiguous", "poisson"]
regime_colors = {"clustered": "#E63946", "ambiguous": "#AAAAAA", "poisson": "#457B9D"}
x_labels = []
bar_data = {r: [] for r in regime_names}

for s in setting_order:
    subset = regime_df[regime_df["tectonic_setting"] == s]
    if len(subset) >= 5:
        x_labels.append(f"{s}\n(n={len(subset)})")
        for r in regime_names:
            bar_data[r].append((subset["regime"] == r).sum() / len(subset) * 100)

x = np.arange(len(x_labels))
bottom = np.zeros(len(x_labels))
for r in regime_names:
    vals = bar_data[r]
    ax.bar(x, vals, bottom=bottom, color=regime_colors[r], label=r, alpha=0.8)
    bottom += np.array(vals)

ax.set_xticks(x)
ax.set_xticklabels(x_labels, rotation=30, ha="right")
ax.set_ylabel("Fraction (%)")
ax.set_title("IET Regime by Tectonic Setting")
ax.legend(fontsize=8)

# Panel 2: Weibull k by tectonic setting (box plot)
ax = axes[1]
plot_data = []
plot_labels = []
box_colors = []
for s in setting_order:
    vals = regime_df[regime_df["tectonic_setting"] == s]["weibull_k"]
    if len(vals) >= 5:
        plot_data.append(vals.values)
        plot_labels.append(f"{s}\n(n={len(vals)})")
        box_colors.append(setting_colors[s])

bp = ax.boxplot(plot_data, tick_labels=plot_labels, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, label="Poisson (k=1)")
ax.set_ylabel("Weibull k")
ax.set_title("Temporal Clustering by Tectonic Setting")
ax.tick_params(axis="x", rotation=30)

# Panel 3: Median IET by tectonic setting
ax = axes[2]
medians = []
labels = []
colors = []
for s in setting_order:
    vals = regime_df[regime_df["tectonic_setting"] == s]["median_iet_hours"]
    if len(vals) >= 5:
        medians.append(vals.median())
        labels.append(f"{s}\n(n={len(vals)})")
        colors.append(setting_colors[s])

ax.bar(range(len(medians)), medians, color=colors, alpha=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Median IET (hours)")
ax.set_title("Median Inter-Event Time by Setting")

fig.suptitle("Inter-Event Time Regime by Tectonic Setting (PB2002)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save_figure(fig, "03_regime_by_tectonic_setting")
plt.show()

# Summary statistics
print("Regime distribution by tectonic setting:")
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations

for s in setting_order:
    subset = regime_df[regime_df["tectonic_setting"] == s]
    if len(subset) >= 5:
        counts = subset["regime"].value_counts()
        k_med = subset["weibull_k"].median()
        print(f"  {s:15s}: n={len(subset):3d}, k_median={k_med:.3f}, "
              f"clustered={counts.get('clustered', 0)}, "
              f"poisson={counts.get('poisson', 0)}, "
              f"ambiguous={counts.get('ambiguous', 0)}")

# Kruskal-Wallis on Weibull k across settings
valid_settings = [s for s in setting_order
                  if len(regime_df[regime_df["tectonic_setting"] == s]) >= 5]
groups = [regime_df[regime_df["tectonic_setting"] == s]["weibull_k"].values
          for s in valid_settings]
H, p = kruskal(*groups)
print(f"\nKruskal-Wallis on Weibull k across settings: H = {H:.1f}, p = {p:.2e}")

# Post-hoc pairwise Mann-Whitney U with Bonferroni correction
if p < 0.05:
    pairs = list(combinations(range(len(valid_settings)), 2))
    n_comparisons = len(pairs)
    print(f"\nPost-hoc pairwise Mann-Whitney U tests ({n_comparisons} comparisons, Bonferroni-corrected):")
    sig_pairs = []
    for i, j in pairs:
        u_stat, u_p = mannwhitneyu(groups[i], groups[j], alternative="two-sided")
        corrected_p = min(u_p * n_comparisons, 1.0)
        sig = "*" if corrected_p < 0.05 else ""
        if corrected_p < 0.05:
            sig_pairs.append((valid_settings[i], valid_settings[j], corrected_p))
        print(f"  {valid_settings[i]:12s} vs {valid_settings[j]:12s}: "
              f"U = {u_stat:.0f}, p_raw = {u_p:.2e}, p_corrected = {corrected_p:.2e} {sig}")
    if not sig_pairs:
        print("  No pairwise comparisons significant after Bonferroni correction.")
    else:
        print(f"  {len(sig_pairs)} pair(s) significant after Bonferroni correction.")

Regime distribution by tectonic setting:
  subduction     : n=297, k_median=0.490, clustered=231, poisson=0, ambiguous=66
  convergent     : n= 59, k_median=0.422, clustered=43, poisson=0, ambiguous=16
  transform      : n= 91, k_median=0.482, clustered=71, poisson=0, ambiguous=20
  spreading      : n= 62, k_median=0.380, clustered=46, poisson=0, ambiguous=16
  rift           : n= 47, k_median=0.450, clustered=33, poisson=0, ambiguous=14
  intraplate     : n=159, k_median=0.483, clustered=104, poisson=0, ambiguous=55

Kruskal-Wallis on Weibull k across settings: H = 13.2, p = 2.16e-02

Post-hoc pairwise Mann-Whitney U tests (15 comparisons, Bonferroni-corrected):
  subduction   vs convergent  : U = 10482, p_raw = 1.72e-02, p_corrected = 2.58e-01 
  subduction   vs transform   : U = 14515, p_raw = 2.85e-01, p_corrected = 1.00e+00 
  subduction   vs spreading   : U = 11117, p_raw = 1.02e-02, p_corrected = 1.53e-01 
  subduction   vs rift        : U = 6883, p_raw = 8.80e-01, p_corrected =

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_5978/20721810.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook classified seismically active 2°×2° grid cells globally into inter-event time (IET) regimes by fitting four parametric distributions (exponential, gamma, log-normal, Weibull) to the **full, non-declustered** USGS ComCat catalog (2000--2025). All fits used MLE with a fixed location parameter (`floc=0`), yielding 1 free parameter for the exponential and 2 for each of the gamma, log-normal, and Weibull models. Model selection was by AIC.

### Regime classification results

- **Clustered (Weibull k < 0.9):** 528 cells (73.8%). Short inter-event times are heavily over-represented relative to a Poisson process, reflecting the dominance of aftershock sequences in the non-declustered catalog.
- **Poisson-like (0.9 <= k <= 1.1):** 0 cells (0.0%). No grid cell met the Poisson classification criteria with the `floc=0` constraint.
- **Quasi-periodic (k > 1.1):** 0 cells identified. No grid cell exhibited quasi-periodic behavior at this spatial and temporal scale.
- **Ambiguous (delta-AIC <= 10):** 187 cells (26.2%). No single distribution achieved a decisive AIC advantage.

### Key statistical findings

- **KS goodness-of-fit tests** assessed absolute model adequacy: 347 adequate fits (48.5%) and 368 poor fits (51.5%) out of 715 cells. The high rejection rate reflects the KS test's sensitivity at large sample sizes rather than catastrophic model failure.
- **Depth is the strongest correlate of clustering intensity:** Spearman rho(k, depth) = 0.619, p = 8.76e-77 — deeper seismicity shows systematically less temporal clustering, consistent with fewer triggered aftershock sequences at depth.
- **b-value correlation is weak but significant:** Spearman rho(k, b-value) = 0.084, p = 0.034 — regions with higher b-values show marginally less clustering.
- **Kruskal-Wallis test** on Weibull k across tectonic settings is significant (H = 13.2, p = 0.022), but no individual pair survives Bonferroni correction across 15 comparisons. Spreading ridges show the most intense clustering (median k = 0.380) and intraplate regions the least (median k = 0.483).
- Strain rate correlates with regime: clustered regions have higher median strain rates (131.5 nanostr/yr) than ambiguous regions (76.0 nanostr/yr).

### Interpretation

The dominance of clustered behavior (73.8%) and complete absence of both Poisson and quasi-periodic cells is a direct and expected consequence of analyzing total seismicity including aftershocks. The large ambiguous fraction (26.2%) — substantially higher than the initial run before the `floc=0` constraint was applied — reflects that many cells lack a decisively best-fitting model when location parameters are properly constrained. The depth correlation (rho = 0.619) is the most striking finding: shallower seismicity is far more temporally clustered than deep seismicity, consistent with the physics of aftershock triggering in the brittle crust.

### Limitations

- **No declustering:** The catalog has not undergone aftershock removal. These results characterize the composite temporal structure of mainshock-aftershock sequences, not background seismicity regime. Any periodic signal from tectonic stress-loading cycles is overwhelmed by aftershock clustering.
- **Marginal sample sizes:** Some grid cells and tectonic-setting groups have limited event counts, which reduces statistical power for distribution fitting and group comparisons.
- **KS test sensitivity:** For cells with large sample sizes, the KS test may reject adequate fits due to minor deviations that are not practically meaningful.

### Future work

- Apply aftershock declustering (e.g., Gardner-Knopoff 1974 or Reasenberg 1985) before regime classification to reveal background temporal structure and potential quasi-periodic behavior in plate boundary regions.
- Compare declustered vs. non-declustered regime maps to quantify the aftershock contribution to temporal clustering and identify regions where background seismicity departs from Poisson behavior.